In [1]:
import sys, os, json
import pandas as pd
sys.path.insert(0, "../datos")
sys.path.insert(0, "..")
from construir import crear_base, esquema_texto, generar_pares, validar_pares, ejecutar_sql
from esquemas import ESQUEMAS, ESQUEMAS_VALIDACION
print("Esquemas de entrenamiento:", list(ESQUEMAS))
print("Esquemas de validacion:", list(ESQUEMAS_VALIDACION))

Esquemas de entrenamiento: ['tienda', 'universidad', 'biblioteca', 'hotel', 'cine', 'gimnasio', 'logistica', 'banco', 'hr', 'music_store', 'restaurante', 'taller']
Esquemas de validacion: ['veterinaria', 'airline']


## 1. Crear las bases en MySQL

Cada base se llena con datos aleatorios usando una semilla fija para que el resultado sea siempre el mismo.

In [2]:
os.makedirs("../datos/esquemas_sql", exist_ok=True)
for nombre, tablas in {**ESQUEMAS, **ESQUEMAS_VALIDACION}.items():
    crear_base(nombre, tablas)
    with open(f"../datos/esquemas_sql/{nombre}.sql", "w") as f:
        f.write(esquema_texto(nombre))
    print(nombre, len(tablas), "tablas")
print(ejecutar_sql("tienda", "SELECT VERSION()")["filas"])

tienda 4 tablas


universidad 5 tablas


biblioteca 4 tablas


hotel 4 tablas


cine 4 tablas


...

music_store 4 tablas


restaurante 4 tablas


taller 4 tablas


veterinaria 6 tablas
airline 5 tablas


[('8.4.10',)]


Asi se ve el esquema que recibe el modelo. Es la salida de SHOW CREATE TABLE con sus COMMENT.

In [3]:
print(esquema_texto("hotel")[:1200])

CREATE TABLE `htl_hab` (
  `hab_id` int NOT NULL AUTO_INCREMENT COMMENT 'identificador de la habitacion',
  `hab_num` varchar(10) NOT NULL COMMENT 'numero de habitacion',
  `hab_tipo` varchar(20) NOT NULL COMMENT 'tipo de habitacion',
  `hab_tarifa` decimal(10,2) NOT NULL COMMENT 'tarifa por noche',
  `hab_piso` int NOT NULL COMMENT 'piso donde esta',
  PRIMARY KEY (`hab_id`)
) ENGINE=InnoDB AUTO_INCREMENT=61 DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_0900_ai_ci COMMENT='habitaciones del hotel';

CREATE TABLE `htl_hues` (
  `hues_id` int NOT NULL AUTO_INCREMENT COMMENT 'identificador del huesped',
  `hues_nom` varchar(80) NOT NULL COMMENT 'nombre del huesped',
  `hues_pais` varchar(40) NOT NULL COMMENT 'pais del huesped',
  PRIMARY KEY (`hues_id`)
) ENGINE=InnoDB AUTO_INCREMENT=121 DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_0900_ai_ci COMMENT='huespedes registrados';

CREATE TABLE `htl_res` (
  `res_id` int NOT NULL AUTO_INCREMENT COMMENT 'identificador de la reserva',
  `hues_id` int NOT NU

## 2. Generar los pares de entrenamiento

Los pares se generan con plantillas escritas por nosotros con ayuda de un LLM. Hay tres niveles. El nivel facil usa una tabla. El nivel medio usa JOIN, GROUP BY y HAVING. El nivel dificil usa subconsultas, WITH, funciones de ventana y fechas. Cada pregunta dice que columnas devolver.

In [4]:
pares = []
for nombre, tablas in ESQUEMAS.items():
    pares += generar_pares(nombre, tablas, 450)
print("Pares generados:", len(pares))

Pares generados: 5400


## 3. Validar en MySQL y guardar descartes

Solo se aceptan pares cuya consulta se ejecuta sin error en MySQL 8.4.10 con la configuracion por defecto, que incluye ONLY_FULL_GROUP_BY. Tambien quitamos preguntas repetidas y consultas con resultado vacio porque no ayudan a aprender.

In [5]:
validos, descartes = validar_pares(pares)
print("Pares validos:", len(validos))
print("Pares descartados:", len(descartes))
pd.DataFrame(descartes)["motivo"].value_counts()

Pares validos: 2630
Pares descartados: 2770


motivo
pregunta duplicada    2734
resultado vacio         36
Name: count, dtype: int64

In [6]:
with open("../datos/entrenamiento.jsonl", "w") as f:
    for p in validos:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
pd.DataFrame(descartes).to_csv("../datos/descartes.csv", index=False)
df = pd.DataFrame(validos)
pd.crosstab(df["base"], df["nivel"], margins=True)

nivel,dificil,facil,media,All
base,,,,
banco,66,86,76,228
biblioteca,43,89,82,214
cine,41,92,74,207
gimnasio,60,90,80,230
hotel,44,92,86,222
hr,67,81,85,233
logistica,49,86,88,223
music_store,44,106,72,222
restaurante,48,91,74,213


In [7]:
for _, fila in df.sample(8, random_state=1).iterrows():
    print(fila["nivel"], "|", fila["pregunta"])
    print("   ", fila["sql"])

facil | ¿Qué bodegas tienen capacidad en metros cubicos entre 3074 y 3945? Devuelve nombre de la bodega y bod_cap.
    SELECT bod_nom, bod_cap FROM lg_bod WHERE bod_cap BETWEEN 3074 AND 3945;
facil | Lista las ventas con fecha de la venta despues del 2023-03-22. Devuelve sale_id y sale_date.
    SELECT sale_id, sale_date FROM sales WHERE sale_date > '2023-03-22';
facil | Lista las facultades cuyo nombre de la facultad empieza con la letra 'I'. Devuelve nombre.
    SELECT nombre FROM facultades WHERE nombre LIKE 'I%';
media | Para cada uno de los socios, calcula el maximo de dias que duro el prestamo de sus prestamos. Devuelve nombre y el resultado.
    SELECT p.nombre, MAX(h.dias) AS resultado FROM socios p JOIN prestamos h ON h.id_socio = p.id_socio GROUP BY p.id_socio, p.nombre;
facil | Lista todos los empleados con sus columnas hire_date y job_title.
    SELECT hire_date, job_title FROM employees;
media | Para cada uno de los libros, calcula el promedio de dias que duro el prestamo 

## 4. Validacion propia

Escribimos a mano 40 preguntas sobre los esquemas `veterinaria` y `airline`. Primero revisamos que ninguna tabla de validacion aparezca en el entrenamiento.

In [8]:
tablas_entr = {t["nombre"] for tablas in ESQUEMAS.values() for t in tablas}
tablas_val = {t["nombre"] for tablas in ESQUEMAS_VALIDACION.values() for t in tablas}
print("Tablas en comun:", tablas_entr & tablas_val)
print("Bases en comun:", set(df["base"]) & set(ESQUEMAS_VALIDACION))

Tablas en comun: set()
Bases en comun: set()


In [9]:
val = json.load(open("../datos/validacion_propia.json"))
filas_ref = [len(ejecutar_sql(v["base"], v["sql"])["filas"]) for v in val]
dv = pd.DataFrame(val)
dv["filas_referencia"] = filas_ref
print("Preguntas de validacion:", len(dv))
print("Consultas de referencia sin resultado:", (dv["filas_referencia"] == 0).sum())
pd.crosstab(dv["base"], dv["nivel"], margins=True)

Preguntas de validacion: 40
Consultas de referencia sin resultado: 0


nivel,dificil,facil,media,All
base,,,,
airline,6,6,7,19
veterinaria,6,7,8,21
All,12,13,15,40


## 5. Largo de los ejemplos

Medimos cuantos tokens tiene cada ejemplo con el prompt final. Esto nos sirve para estimar el tiempo de entrenamiento.

In [10]:
import solucion
from transformers import AutoTokenizer
solucion.tokenizer = AutoTokenizer.from_pretrained(os.environ.get("RUTA_MODELO", solucion.CONFIG["modelo_base"]))
esquemas = {b: open(f"../datos/esquemas_sql/{b}.sql").read() for b in df["base"].unique()}
largos = [len(solucion.tokenizer(solucion.armar_prompt(solucion.armar_mensajes(esquemas[p["base"]], p["pregunta"])) + p["sql"])["input_ids"]) for p in validos]
pd.Series(largos).describe()

count    2630.000000
mean      489.748669
std        51.254891
min       396.000000
25%       453.000000
50%       482.000000
75%       513.000000
max       678.000000
dtype: float64

## Resumen

Generamos mas de 2,000 pares validos sobre 12 esquemas propios. Los descartes se deben a preguntas repetidas que salen de las plantillas y a consultas con resultado vacio. Ninguna consulta valida falla en MySQL 8.4.10. Los dos esquemas de validacion no comparten tablas con el entrenamiento.